<a id="top"></a>
# Finding JWST L2 Constituents for an L3 Data Product

This notebook demonstrates data level traversal for James Webb Space Telescope (JWST) data from L3 to L2.

Data processed by mission pipelines at STScI have increasing *product levels* as they are calibrated and reduced. Level 2 (L2) data products are count-rate images with a data quality bitmask. L2 products are combined and resampled to create Level 3 (L3) products, which are flux calibrated science products. L2 and L3 data products are described in detail in for JWST in [JDox](https://jwst-pipeline.readthedocs.io/en/latest/jwst/data_products/stages.html) and for Roman in [RDox](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products).

User Story: 
Suppose you have the footprint of a JWST L3 observation from `astroquery` loaded in `mast-aladin`. You want to see the L2 datasets that were used to produce this L3. To do so, you leverage association files to see the footprints of the constituent L2s used to produce the L3.

***

## Initial setup and imports

In [ ]:
# Imports
from astroquery.mast import MastMissions
import numpy as np
import json

from mast_aladin import MastAladin

# Create a MastMissions object, assign mission to 'jwst' for James Webb Space Telescope data
jwst_mission = MastMissions(mission='jwst')

# Set up mast-aladin for viewing footprints
mast_aladin = MastAladin(
    target = "05 34 31.78 +22 01 02.6",
    fov = 1,
    height = 800,
)
mast_aladin.show()

## Running search for L3 products

In [ ]:
# Query for data level 3 products around the Crab Nebula
results = jwst_mission.query_object('CRAB NEBULA',radius=1, productLevel=3)

In [ ]:
# Add the L3 results table from initial query for display
L3_table = mast_aladin.add_table(results)

In [ ]:
# Determine what products are available for the first result
products = jwst_mission.get_product_list(results[0])

In [ ]:
# Download the association file(s) for the first result. Defaults to
# download to the local path `./mastDownload/jwst/{filename}
download = jwst_mission.download_products(products, extension = "json", cache = True)

## Determine associated L2s

In [ ]:
# Parse the downloaded association file(s) from above
associated_L2s = []

for assoc_file in download["Local Path"]:
    with open(assoc_file, 'r') as file:
         data = json.load(file)
    associated_L2s.extend([x["expname"] for x in data["products"][0]["members"]])

# Remove ends of strings from L2 file-naming convention to match fileSetName in jwst MastMissions results table
associated_L2s = [x.removesuffix("_mirimage_cal.fits") for x in associated_L2s]

In [ ]:
# Query the same object with larger radius to capture all potential associated L2s
followup_results = jwst_mission.query_object('CRABNEBULA',radius=5)

# Filter to exclude L3 results
filtered = followup_results[followup_results["productLevel"] != "3"]

# Create a mask to grab the L2s of interest we found above
mask = np.isin(filtered["fileSetName"], associated_L2s)

# Apply the mask to get the filtered table, make sure we have the same number of results
filtered_data = filtered[mask]
len(filtered_data), len(associated_L2s)

In [ ]:
# Lastly, add those into our aladin viewer! We can see the L2s comprising the initial L3 in the aladin viewer now
level2_table = mast_aladin.add_table(filtered_data)

## Additional resources

- [mast-aladin Documentation](https://mast-aladin.readthedocs.io/en/latest/index.html)
- [JDox Data Levels and Products](https://jwst-pipeline.readthedocs.io/en/latest/jwst/data_products/stages.html)
- [RDox WFI Data Levels and Products](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/data-levels-and-products)

## About this notebook
**Author:** Celia Parts

**Last updated:** October 2026
***
[Top of Page](#top)
<img style="float: right;" src="https://raw.githubusercontent.com/spacetelescope/style-guides/master/guides/images/stsci-logo.png" alt="Space Telescope Logo" width="200px"/> 